# M002 – Grade Performance Mart Validation

## Objective

This notebook validates the Grade Performance Mart built from the LendingClub feature store.

The purpose of this mart is to provide a portfolio-level view of credit performance by LendingClub grade (A–G). Unlike the feature store, which contains one row per loan, this mart summarizes millions of loans into seven grade segments that can be used for portfolio monitoring, risk reporting, pricing analysis, and future model validation.

The validation process confirms:

- Correct mart creation
- Correct mart grain
- Reconciliation to the feature store
- Correct aggregation logic
- Preservation of portfolio default rates
- Business reasonableness of grade segmentation

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

DB_PATH = (
    Path.cwd().parent.parent
    / "data"
    / "warehouse"
    / "duckdb"
    / "lendingclub.duckdb"
)

print(DB_PATH)

conn = duckdb.connect(str(DB_PATH))

d:\project_lighthouse\projects\P0_Data_Platform\datasets\lendingclub\data\warehouse\duckdb\lendingclub.duckdb


In [2]:
conn.execute('select count(*) as mart_rows from mart.grade_performance').fetchdf()

,mart_rows
0,7


## Validation Result

The mart contains 7 rows.

This is expected because the mart is aggregated at the grade level and LendingClub grades range from A through G.

The row count immediately confirms that aggregation occurred correctly and that the mart is no longer operating at individual-loan granularity.

## Grain Validation

The intended grain of this mart is:

**One row per LendingClub grade**

The following validation confirms that each grade appears only once in the mart.

In [3]:
conn.execute('select count(*) as rows, count(distinct grade) as unique_grades from mart.grade_performance').fetchdf()

,rows,unique_grades
0,7,7


## Validation Result

The mart contains 7 rows and 7 unique grades.

This confirms that the mart grain is correctly implemented with no duplicate grade records.

The mart can therefore be safely used for reporting and downstream analysis without double-counting exposure or defaults.

## Loan Count Reconciliation

The total number of loans represented in the mart should exactly match the total number of loans contained in the feature store.

This reconciliation ensures that no records were lost or duplicated during aggregation.

In [4]:
conn.execute('''select (select count(*) from feature.loan_features_v1) feature_rows,(select sum(loan_count) from mart.grade_performance) mart_loan_count''').fetchdf()

,feature_rows,mart_loan_count
0,2260668,2260668.0


## Validation Result

The total loan count in the mart exactly matches the total loan count in the feature store.

This is a critical control because it confirms that the mart represents the entire LendingClub population and that aggregation logic has not introduced data loss.

## Schema Review

The schema review confirms that all expected business metrics were successfully created during mart construction.

These metrics are intended to support portfolio monitoring, risk analysis, and executive reporting.

In [5]:
conn.execute('describe mart.grade_performance').fetchdf()

,column_name,column_type,null,key,default,extra
0,grade,VARCHAR,YES,None,None,None
1,loan_count,BIGINT,YES,None,None,None
2,total_loan_amount,DOUBLE,YES,None,None,None
3,avg_loan_amount,DOUBLE,YES,None,None,None
4,avg_interest_rate,DOUBLE,YES,None,None,None
5,avg_fico,DOUBLE,YES,None,None,None
6,avg_dti,DOUBLE,YES,None,None,None
7,avg_credit_history_years,DOUBLE,YES,None,None,None
8,avg_loan_to_income_ratio,DOUBLE,YES,None,None,None
9,avg_revolving_utilization,DOUBLE,YES,None,None,None


## Validation Result

The mart contains metrics covering:

- Portfolio size
- Loan exposure
- Interest rate characteristics
- Credit quality
- Borrower leverage
- Credit history
- Default performance
- Utilization behaviour

The resulting structure provides a compact portfolio summary that can support both management reporting and future credit risk analytics initiatives.

## Default Rate Reconciliation

Default rate is one of the most important measures within the warehouse.

The portfolio default rate calculated directly from the feature store should match the weighted default rate calculated from the mart.

In [6]:
conn.execute('''select (select round(avg(default_flag)*100,4) from feature.loan_features_v1) feature_default_rate,(select round(sum(default_count)*100.0/sum(loan_count),4) from mart.grade_performance) mart_default_rate''').fetchdf()

,feature_default_rate,mart_default_rate
0,12.8646,12.8646


## Validation Result

The portfolio default rate from the feature store exactly matches the portfolio default rate reconstructed from the mart.

This confirms that default counts and loan counts were aggregated correctly and that the mart preserves portfolio-level risk statistics without distortion.

## Grade Performance Analysis

The following table represents the final output of the mart.

Each row summarizes all LendingClub loans belonging to a particular grade and provides a consolidated view of risk, pricing, borrower quality, and portfolio performance.

In [7]:
conn.execute('select * from mart.grade_performance order by grade').fetchdf()

,grade,loan_count,total_loan_amount,avg_loan_amount,avg_interest_rate,avg_fico,avg_dti,avg_credit_history_years,avg_loan_to_income_ratio,avg_revolving_utilization,default_count,default_rate,high_utilization_count,high_utilization_rate
0,A,433027,6.323642e+09,14603.343210,7.084545,730.992703,16.238648,17.890627,0.527162,37.064114,15536.0,3.59,20784.0,4.80
1,B,663557,9.404818e+09,14173.338199,10.675806,701.831470,17.966630,16.708781,0.546628,48.943646,57449.0,8.66,75968.0,11.45
2,C,650053,9.775551e+09,15038.083318,14.143689,691.274291,19.552054,15.917377,0.551594,54.435478,93359.0,14.36,106014.0,16.31
3,D,324424,5.097344e+09,15711.983007,18.143067,685.956383,20.929692,15.404252,0.697766,57.443781,66025.0,20.35,64554.0,19.90
4,E,135639,2.367318e+09,17453.078392,21.829653,684.421169,21.549435,15.309605,0.764530,59.132823,38365.0,28.28,30504.0,22.49
5,F,41800,7.994102e+08,19124.646531,25.454091,682.383744,21.676804,14.996380,0.329392,60.039823,15225.0,36.42,10249.0,24.52
6,G,12168,2.480324e+08,20383.988741,28.074255,681.238577,22.432815,14.834198,0.325622,59.283907,4868.0,40.01,2925.0,24.04


## Key Findings

Several important portfolio characteristics are visible.

### 1. Default Rates Increase Consistently Across Grades

Default rates rise from:

- Grade A: 3.59%
- Grade B: 8.66%
- Grade C: 14.36%
- Grade D: 20.35%
- Grade E: 28.28%
- Grade F: 36.42%
- Grade G: 40.01%

This monotonic relationship demonstrates that the LendingClub grading system successfully separates lower-risk borrowers from higher-risk borrowers.

### 2. FICO Scores Decline As Risk Increases

Average FICO decreases from approximately 731 in Grade A to approximately 681 in Grade G.

This behaviour is economically intuitive and provides further validation that grade assignments align with borrower credit quality.

### 3. Interest Rates Increase With Risk

Average interest rates rise steadily from approximately 7% for Grade A loans to approximately 28% for Grade G loans.

This indicates that pricing generally compensates for increasing credit risk.

### 4. Utilization Increases With Risk

High-utilization behaviour becomes increasingly common as grade quality deteriorates.

This relationship supports the predictive value of utilization-based risk features created in the feature store.

### 5. Portfolio Concentration

The majority of LendingClub loans are concentrated within Grades B and C.

These segments therefore represent the largest contributors to both portfolio growth and portfolio risk.

## Loan-to-Income Ratio Investigation

During mart review, unusually high average loan-to-income ratios were observed in Grades D and E.

The following analysis investigates whether these averages are being influenced by extreme observations.

In [8]:
# Loan To Income Ratio Validation

conn.execute("""
    select

        grade,

        round(min(loan_to_income_ratio), 4) as min_lti,

        round(avg(loan_to_income_ratio), 4) as avg_lti,

        round(max(loan_to_income_ratio), 4) as max_lti

    from feature.loan_features_v1

    group by grade

    order by grade
    """).fetchdf()

conn.close()

## Investigation Findings

The maximum loan-to-income ratios are extremely large across several grades, reaching values far beyond economically realistic borrowing behaviour.

Examples include values above:

- 24,000
- 32,000
- 40,000

These observations indicate the presence of extreme outliers caused by very small reported incomes or data quality anomalies.

As a result:

- Minimum values appear reasonable.
- Average values are heavily influenced by outliers.
- Median-based metrics or capped ratios may be preferable for future analytical use.

### Decision

The current feature remains acceptable for P0 because the objective is warehouse construction rather than production model development.

However, a future version of the feature store should introduce capping or winsorization rules for loan-to-income ratios before modelling activities begin.

# Validation Conclusion

The Grade Performance Mart has successfully passed all validation checks.

### Controls Passed

✓ Mart created successfully

✓ Correct grain (one row per grade)

✓ Unique grade records

✓ Loan count reconciliation

✓ Default rate reconciliation

✓ Business logic validation

✓ Risk ranking validation

### Business Conclusions

The LendingClub grading framework demonstrates strong risk segmentation characteristics:

- Higher grades have higher FICO scores.
- Higher grades receive lower interest rates.
- Higher grades experience lower default rates.
- Credit quality deteriorates consistently from Grade A to Grade G.

The mart therefore provides a reliable portfolio-level view of LendingClub credit risk and is suitable for reporting, monitoring, and future credit risk analytics use cases.

### P0 Status

M002 – Grade Performance Mart is complete and validated.
